# SSTR2–SST14 펩타이드 설계 파이프라인

**Somatostatin Receptor 2 (SSTR2)** 에 결합하는 **SST-14 유도체 펩타이드**를 계산적으로 설계하는 파이프라인입니다.

### 파이프라인 개요

```
AlphaFold3 예측 구조 (CIF)
  ↓ CIF → PDB 변환
  ↓ 체인 표준화 (A=수용체, B=펩타이드)
  ↓ Peptide-only Relax (에너지 최소화)
  ↓ FastDesign × 20 (서열 설계)
  ↓ 필터링 (Cys 이황화결합 보존 검증)
  ↓ FlexPepDock Refine (결합 정밀화)
  ↓ Top-3 후보 구조 3D 시각화
```

### 배경

- **SSTR2**: G-protein coupled receptor (GPCR). Somatostatin을 감지하여 성장호르몬·인슐린 등의 분비를 억제
- **SST-14**: 14잔기 환형 펩타이드 (`AGCKNFFWKTFTSC`). Cys3–Cys14 이황화결합으로 환형 구조 유지
- **목표**: SST-14의 결합력을 유지/개선하면서 안정성·약동학적 특성을 개선한 변이체 설계

### 필요 사항

- **PyRosetta 라이선스**: [pyrosetta.org](https://www.pyrosetta.org) 에서 학술용 라이선스 발급 가능
- **실행 시간**: 전체 파이프라인 약 2–3시간 (FastDesign + FlexPepDock)
- **입력 파일**: `input/fold_test1_model_0.cif` (AlphaFold3 예측 결과, 폴더에 포함)

---
## 0. 환경 설정

In [ ]:
#@title 0-1. 패키지 설치 (최초 1회만 실행) { display-mode: "form" }
!pip install -q pyrosetta-installer biopython py3Dmol pandas tqdm
import pyrosetta_installer; pyrosetta_installer.install_pyrosetta()

PyRosetta install detected, doing nothing...


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
%cd /content/drive/MyDrive/Colab Notebooks/colab_sstr2_demo

/content/drive/MyDrive/Colab Notebooks/colab_sstr2_demo


In [ ]:
#@title 0-2. 라이브러리 로드 { display-mode: "form" }
import os, sys, time, json
import pandas as pd
import py3Dmol
from IPython.display import display, HTML
from tqdm.notebook import tqdm

pd.set_option("display.max_colwidth", 200)
pd.set_option("display.width", 140)

# PyRosetta
import pyrosetta
pyrosetta.init("-mute all -relax:default_repeats 3")

# 파이프라인 헬퍼 함수
from utils.pipeline_helpers import *

print("All libraries loaded.")

┌───────────────────────────────────────────────────────────────────────────────┐
│                                  PyRosetta-4                                  │
│               Created in JHU by Sergey Lyskov and PyRosetta Team              │
│               (C) Copyright Rosetta Commons Member Institutions               │
│                                                                               │
│ NOTE: USE OF PyRosetta FOR COMMERCIAL PURPOSES REQUIRES PURCHASE OF A LICENSE │
│          See LICENSE.PyRosetta.md or email license@uw.edu for details         │
└───────────────────────────────────────────────────────────────────────────────┘
PyRosetta-4 2026 [Rosetta PyRosetta4.Release.python312.ubuntu 2026.06+release.e5a76a2dbd7e42e1271d0af460926c86e29e59d1 2026-02-02T18:21:37] retrieved from: http://www.pyrosetta.org
All libraries loaded.


---
## 1. 입력 구조 확인

AlphaFold3가 예측한 **SSTR2–SST14 복합체** 구조를 로드합니다.

- AlphaFold3는 수용체(SSTR2)와 펩타이드(SST-14)의 결합 구조를 예측
- 출력 포맷인 CIF를 PyRosetta가 인식하는 PDB로 변환
- 체인별 잔기 수, 서열, 범위를 확인하여 구조가 올바르게 로드되었는지 검증

In [ ]:
INPUT_CIF = "input/fold_test1_model_0.cif"
OUTPUT_PDB = "input/fold_test1_model_0.pdb"

assert os.path.exists(INPUT_CIF), f"CIF 파일이 없습니다: {INPUT_CIF}"

INPUT_PDB = cif_to_pdb(INPUT_CIF, OUTPUT_PDB)
print(f"CIF -> PDB 변환 완료: {INPUT_PDB}")

CIF -> PDB 변환 완료: input/fold_test1_model_0.pdb


In [ ]:
summarize_pdb(INPUT_PDB, show_seq="head")

,chain,length,pdb_res_min,pdb_res_max,ranges,seq
0,A,14,1,14,1-14,AGCKNFFWKTFTSC
1,B,369,1,369,1-369,MDMADEPLNGSHTWLSIPFDLNGSVVSTNTSNQTEPYYDLTSNAVLTFIYFVVCIIGLCG...


In [ ]:
show_structure_3d(INPUT_PDB, label="SSTR2-SST14 (AlphaFold3)")

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

<IPython.core.display.HTML object>

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

---
## 2. 구조 표준화

PyRosetta의 프로토콜들은 체인 순서에 민감합니다. 파이프라인 전체에서 일관된 결과를 얻기 위해:

- **Chain A** = 수용체 (SSTR2, ~390 잔기)
- **Chain B** = 펩타이드 (SST-14, 14 잔기)

으로 표준화합니다. 펩타이드는 **길이 = 14**인 체인을 자동 탐지합니다.

In [ ]:
pose = pyrosetta.pose_from_pdb(INPUT_PDB)
peptide_chain_id = find_peptide_chain_pose(pose, peptide_len=14)
print(f"Peptide chain detected: chain {peptide_chain_id}")

,pose_chain_id,length,sequence
0,1,14,AGCKNFFWKTFTSC
1,2,369,MDMADEPLNGSHTWLSIPFDLNGSVVSTNTSNQTEPYYDLTSNAVLTFIYFVVCIIGLCGNTLVIYVILRYAKMKTITNIYILNLAIADELFMLGLPFLAMQVALVHWPFGKAICRVVMTVDGINQFTSIFCLTVMSIDRYLAVVHPIKSAKWRRPRTAKMITMAVWGVSLLVILPIMIYAGLRSNQWGRSSCTIN...


Peptide chain detected: chain 1


In [ ]:
standard_pose = standardize_to_AB(pose, peptide_chain_id, out_pdb="standardized_raw.pdb")

[OK] standardized saved -> standardized_raw.pdb (A=receptor, B=peptide)


In [ ]:
summarize_pdb("standardized_raw.pdb", show_seq="head")

,chain,length,pdb_res_min,pdb_res_max,ranges,seq
0,A,369,1,369,1-369,MDMADEPLNGSHTWLSIPFDLNGSVVSTNTSNQTEPYYDLTSNAVLTFIYFVVCIIGLCG...
1,B,14,370,383,370-383,AGCKNFFWKTFTSC


In [ ]:
show_structure_3d("standardized_raw.pdb", label="Standardized (A=receptor, B=peptide)")

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

<IPython.core.display.HTML object>

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

---
## 3. 펩타이드 Relax (에너지 최소화)

AlphaFold3 예측 구조는 실험적 구조와 약간의 에너지 차이가 있을 수 있습니다.
**FastRelax**를 통해 펩타이드의 백본(backbone)과 측쇄(sidechain)를 Rosetta 에너지 함수 기준으로 최적화합니다.

- **수용체는 고정** (수용체 구조 변경 방지)
- **펩타이드만 완화** (백본 + 측쇄 자유도 허용)
- Relax 후 Rosetta 에너지 점수가 낮아져야 (음수 방향) 정상

> 약 2–5분 소요됩니다.

In [ ]:
pre_score, post_score = relax_peptide_only(
    "standardized_raw.pdb", "standardized_relaxed.pdb", peptide_chain_number=2
)

Peptide-only FastRelax 시작 (score before: 49.00)


FastRelax:   0%|          | 00:00<?

Relax 완료 (48.7s) -> standardized_relaxed.pdb
  score: 49.00 -> -308.27 (delta=-357.26)


In [ ]:
show_comparison_3d(
    ["standardized_raw.pdb", "standardized_relaxed.pdb"],
    labels=["Before Relax", "After Relax"],
    width=900, height=400
)

---
## 4. FastDesign — 펩타이드 서열 설계

Rosetta **FastDesign**을 사용하여 SST-14 서열의 **변이체 후보**를 생성합니다.

### 설계 전략

| 항목 | 설명 |
|------|------|
| **설계 가능 위치** | Cys3, Cys14를 **제외**한 12개 위치 (자동 탐지) |
| **고정 위치** | Cys3–Cys14 (이황화결합 보존 필수) |
| **수용체** | 완전 고정 (PreventRepacking) |
| **후보 수** | 20개 |

### Cys3–Cys14 이황화결합

SST-14의 핵심 구조적 특징입니다. 이 결합이 14잔기 펩타이드를 **환형으로 고정**하여:
- 백본 자유도가 극도로 제한됨
- 각 위치에서 물리적으로 허용되는 측쇄(rotamer)가 제한됨
- 이론적 20^12 조합 중 에너지적으로 허용되는 서열 공간은 훨씬 좁음

> 약 1–2시간 소요됩니다 (20개 후보 × 후보당 3–5분).

In [ ]:
df_candidates, original_pep, design_positions = fastdesign_candidates(
    input_pdb="standardized_relaxed.pdb",
    n=20
)
print(f"Original peptide: {original_pep}")
print(f"Design positions: {design_positions}")

  Auto-detected Cys at positions [3, 14] -> excluded from design
  Design positions: [1, 2, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13]


FastDesign:   0%|          | 0/20 [00:00<?, ?candidate/s]

  TaskFactory: design=[1, 2, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13] (abs=[370, 371, 373, 374, 375, 376, 377, 378, 379, 380, 381, 382]), repack_only=[], frozen=receptor+Cys
  TaskFactory: design=[1, 2, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13] (abs=[370, 371, 373, 374, 375, 376, 377, 378, 379, 380, 381, 382]), repack_only=[], frozen=receptor+Cys
  TaskFactory: design=[1, 2, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13] (abs=[370, 371, 373, 374, 375, 376, 377, 378, 379, 380, 381, 382]), repack_only=[], frozen=receptor+Cys
  TaskFactory: design=[1, 2, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13] (abs=[370, 371, 373, 374, 375, 376, 377, 378, 379, 380, 381, 382]), repack_only=[], frozen=receptor+Cys
  TaskFactory: design=[1, 2, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13] (abs=[370, 371, 373, 374, 375, 376, 377, 378, 379, 380, 381, 382]), repack_only=[], frozen=receptor+Cys
  TaskFactory: design=[1, 2, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13] (abs=[370, 371, 373, 374, 375, 376, 377, 378, 379, 380, 381, 382]), repack_only=[], frozen=receptor+Cy

In [ ]:
df_candidates[["candidate", "seq", "dG_REU", "dSASA", "seq_distance",
               "rank_score", "is_unique"]].head(10)

### 결과 해석

| 칸럼 | 의미 |
|--------|------|
| **seq** | 설계된 펩타이드 서열 |
| **dG_REU** | 결합 자유 에너지 (Rosetta Energy Unit). **음수일수록 강한 결합** |
| **dSASA** | 결합 시 묻히는 표면적 (Å²). **클수록 넓은 접촉면** |
| **seq_distance** | 원래 SST-14 대비 변이 잔기 수 |
| **rank_score** | 종합 점수 (dG + 안정성/PK 보정). **높을수록 우수** |

#### 서열이 유사한 이유

FastDesign 결과에서 후보들의 서열이 매우 유사한 것은 정상입니다:

1. **환형 백본 강직성**: Cys3–Cys14가 구조를 고정하여 허용 가능한 측쇄가 제한됨
2. **약효단(pharmacophore) 위치의 에너지 지배**: Trp8, Lys9 등 결합 포켓 깊숙이 접촉하는 잔기는 대체가 어려움
3. **동일 출발 구조**: Monte Carlo 샘플링이 동일한 국소 최소값으로 수렴

---
## 5. 필터링 + FlexPepDock 정제

### 필터링: Cys 이황화결합 보존 검증

FastDesign이 Cys 위치를 변이시킨 후보는 이황화결합이 깨질 위험이 있으므로 제거합니다.

### FlexPepDock Refinement

**FlexPepDock**은 수용체–펩타이드 복합체의 결합 모드(binding mode)를 정밀화하는 전용 프로토콜입니다:

- 펩타이드의 rigid-body 위치/방향을 최적화
- 펩타이드 백본 유연성(flexibility) 허용
- 측쇄 repacking을 통해 인터페이스 최적화

> 약 30분–1시간 소요됩니다 (Top 10 후보 기준).

In [ ]:
df_pass, df_fail = filter_candidates(df_candidates)

if len(df_pass) == 0:
    print("\nCys 변이 통과 후보가 0건 -> 전체 후보를 fallback으로 사용")
    df_pass = df_candidates

display(df_pass[["candidate", "seq", "dG_REU", "dSASA", "rank_score",
                  "seq_distance", "is_unique"]].head(10))

In [ ]:
df_refined = flexpepdock_refine(df_pass, topk=min(10, len(df_pass)))
df_refined

---
## 6. 최종 후보 3D 시각화

FlexPepDock 정제 후 **rank_score 상위 3개** 후보를 3D로 확인합니다.

- 파란색: 수용체 (SSTR2) cartoon
- 주황색: 펩타이드 cartoon + stick
- 마우스로 회전/줌 가능

In [ ]:
topk = min(3, len(df_refined))
if topk == 0:
    print("Refined 후보가 없습니다.")
else:
    top3 = df_refined.head(topk)

    # Best 후보 단독 뷰 (수용체 표면 포함)
    show_structure_3d(
        top3.iloc[0]["pdb_path"], width=800, height=500,
        surface_receptor=True,
        label=f"Best: {top3.iloc[0]['output']} (dG={top3.iloc[0]['dG_REU']:.1f})"
    )

In [ ]:
if topk > 1:
    paths = top3["pdb_path"].tolist()
    labels = [
        f"#{i+1} {row['output']}\ndG={row['dG_REU']:.1f} dSASA={row['dSASA']:.0f}"
        for i, (_, row) in enumerate(top3.iterrows())
    ]
    show_comparison_3d(paths, labels=labels, width=900, height=400)
else:
    print("후보가 1개라 그리드 비교를 건너뗀니다.")

---
## 7. 결과 요약

### 파이프라인 요약

| 단계 | 설명 | 출력 |
|------|------|------|
| 1. 입력 | AlphaFold3 CIF → PDB | `input/fold_test1_model_0.pdb` |
| 2. 표준화 | Chain A=receptor, B=peptide | `standardized_raw.pdb` |
| 3. Relax | 펩타이드 에너지 최소화 | `standardized_relaxed.pdb` |
| 4. FastDesign | 서열 설계 × 20 | `candidates/candidate_*.pdb` |
| 5. 필터링 | Cys 이황화결합 보존 검증 | - |
| 6. FlexPepDock | 결합 모드 정제 | `refined/refined_*.pdb` |

### 다음 단계 (제안)

1. **실험적 검증**: 상위 후보 합성 → binding assay (SPR, FP 등)
2. **MD 시뮬레이션**: 결합 안정성 확인 (GROMACS, OpenMM)
3. **ADMET 예측**: 혁장 안정성, 투과성, 대사 예측
4. **서열 공간 확장**: 더 많은 시드/재시도, 또는 ProteinMPNN 등 ML 기반 설계

In [ ]:
if len(df_refined) > 0:
    print("=" * 60)
    print("SSTR2-SST14 Peptide Design Results")
    print("=" * 60)
    print(f"Original peptide: {original_pep}")
    print(f"Design positions: {design_positions}")
    print(f"\nTop candidates (by rank_score):")
    for i, (_, row) in enumerate(df_refined.head(5).iterrows()):
        print(f"  #{i+1} {row['output']}: seq={row['seq']} "
              f"dG={row['dG_REU']:.1f} dSASA={row['dSASA']:.0f} "
              f"rank={row['rank_score']:.1f}")
    print("=" * 60)